In [1]:
# !pip install pymupdf llama-cpp-python

In [2]:
import os
import re
import sys
import json
import time
import math
import platform
from pathlib import Path

import pandas as pd
import fitz  # PyMuPDF

ROOT = Path.cwd()
SAMPLE_DIR = ROOT / "sample"
MODEL_DIR = ROOT / "models"
SAMPLE_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("OS:", platform.platform())
print("CPU threads visible:", os.cpu_count())
print("Working directory:", ROOT)


Python: 3.12.13
OS: Windows-11-10.0.26200-SP0
CPU threads visible: 8
Working directory: d:\FinGPT


c:\Users\r2000\anaconda3\envs\fingpt\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


## 1. Create our sample statement

This cell creates a fake statement PDF and a CSV ground-truth file. The CSV is only for evaluation; the model will see the PDF text, not the answer key.

We intentionally include:
- repeated Amazon transactions,
- two SIP/investment transactions,
- salary and refund credits,
- ordinary debits,
- a statement period.

This gives us several test questions with exact numerical answers.


In [3]:
sample_rows = [
    ("2026-10-01", "AMAZON PAY INDIA", 1299.00, "DEBIT"),
    ("2026-10-02", "HDFC MUTUAL FUND SIP", 5000.00, "DEBIT"),
    ("2026-10-03", "SALARY ACME TECHNOLOGIES", 90000.00, "CREDIT"),
    ("2026-10-04", "SWIGGY", 450.00, "DEBIT"),
    ("2026-10-05", "UPI FROM FRIEND", 1200.00, "CREDIT"),
    ("2026-10-08", "AMAZON SELLER", 799.00, "DEBIT"),
    ("2026-10-10", "RENT TRANSFER", 15000.00, "DEBIT"),
    ("2026-10-15", "ZERODHA SIP NACH", 3000.00, "DEBIT"),
    ("2026-10-18", "UBER INDIA", 320.00, "DEBIT"),
    ("2026-10-20", "RESTAURANT PAYMENT", 850.00, "DEBIT"),
    ("2026-10-22", "AMAZON REFUND", 499.00, "CREDIT"),
    ("2026-10-25", "ELECTRICITY BILL", 1800.00, "DEBIT"),
]

gt = pd.DataFrame(sample_rows, columns=["date", "description", "amount", "direction"])
gt["date"] = pd.to_datetime(gt["date"])
gt.to_csv(SAMPLE_DIR / "ground_truth_transactions.csv", index=False)

lines = [
    "SAMPLE BANK STATEMENT — SYNTHETIC DATA ONLY",
    "Account holder: Test User",
    "Statement period: 01-Oct-2026 to 31-Oct-2026",
    "Currency: INR",
    "",
    "DATE | DESCRIPTION | AMOUNT (INR) | TYPE",
]
for row in sample_rows:
    date, desc, amount, direction = row
    lines.append(f"{date} | {desc} | {amount:,.2f} | {direction}")

statement_text = "\\n".join(lines)

pdf_path = SAMPLE_DIR / "sample_bank_statement.pdf"
doc = fitz.open()
page = doc.new_page()
page.insert_textbox(
    fitz.Rect(45, 45, 550, 790),
    statement_text,
    fontsize=10,
    fontname="helv",
    lineheight=1.5
)
doc.save(pdf_path)
doc.close()

print("Created:", pdf_path)
print("Created:", SAMPLE_DIR / "ground_truth_transactions.csv")
display(gt)


Created: d:\FinGPT\sample\sample_bank_statement.pdf
Created: d:\FinGPT\sample\ground_truth_transactions.csv


,date,description,amount,direction
0,2026-10-01,AMAZON PAY INDIA,1299.0,DEBIT
1,2026-10-02,HDFC MUTUAL FUND SIP,5000.0,DEBIT
2,2026-10-03,SALARY ACME TECHNOLOGIES,90000.0,CREDIT
3,2026-10-04,SWIGGY,450.0,DEBIT
4,2026-10-05,UPI FROM FRIEND,1200.0,CREDIT
5,2026-10-08,AMAZON SELLER,799.0,DEBIT
6,2026-10-10,RENT TRANSFER,15000.0,DEBIT
7,2026-10-15,ZERODHA SIP NACH,3000.0,DEBIT
8,2026-10-18,UBER INDIA,320.0,DEBIT
9,2026-10-20,RESTAURANT PAYMENT,850.0,DEBIT


## 2. Extract text from the PDF

This is the document-ingestion step. For this synthetic text-based PDF, PyMuPDF should extract the rows cleanly.

Real bank PDFs can have columns, wrapped descriptions, scanned images, or inconsistent layouts. We'll handle bank-specific parsing in a later session.


In [4]:
pdf = fitz.open(pdf_path)
extracted_pages = [page.get_text("text") for page in pdf]
document_text = "\\n".join(extracted_pages)
pdf.close()

print(document_text)


SAMPLE BANK STATEMENT ? SYNTHETIC DATA ONLY\nAccount holder: Test User\nStatement period:
01-Oct-2026 to 31-Oct-2026\nCurrency: INR\n\nDATE | DESCRIPTION | AMOUNT (INR) | TYPE\n2026-10-01 |
AMAZON PAY INDIA | 1,299.00 | DEBIT\n2026-10-02 | HDFC MUTUAL FUND SIP | 5,000.00 |
DEBIT\n2026-10-03 | SALARY ACME TECHNOLOGIES | 90,000.00 | CREDIT\n2026-10-04 | SWIGGY | 450.00 |
DEBIT\n2026-10-05 | UPI FROM FRIEND | 1,200.00 | CREDIT\n2026-10-08 | AMAZON SELLER | 799.00 |
DEBIT\n2026-10-10 | RENT TRANSFER | 15,000.00 | DEBIT\n2026-10-15 | ZERODHA SIP NACH | 3,000.00 |
DEBIT\n2026-10-18 | UBER INDIA | 320.00 | DEBIT\n2026-10-20 | RESTAURANT PAYMENT | 850.00 |
DEBIT\n2026-10-22 | AMAZON REFUND | 499.00 | CREDIT\n2026-10-25 | ELECTRICITY BILL | 1,800.00 |
DEBIT



## 3. Inspect extraction quality

Before asking an LLM, verify that the document was actually extracted.

This quick check is not a full parser. It checks that key transaction descriptions survived extraction.


In [5]:
required_markers = [
    "AMAZON PAY INDIA",
    "HDFC MUTUAL FUND SIP",
    "SALARY ACME TECHNOLOGIES",
    "ZERODHA SIP NACH",
    "AMAZON REFUND",
]

extraction_checks = {
    marker: marker in document_text
    for marker in required_markers
}
display(pd.DataFrame(
    [{"marker": k, "found": v} for k, v in extraction_checks.items()]
))

extraction_rate = sum(extraction_checks.values()) / len(extraction_checks)
print(f"Simple extraction marker coverage: {extraction_rate:.1%}")
assert extraction_rate == 1.0, "PDF extraction missed one or more expected markers."


,marker,found
0,AMAZON PAY INDIA,True
1,HDFC MUTUAL FUND SIP,True
2,SALARY ACME TECHNOLOGIES,True
3,ZERODHA SIP NACH,True
4,AMAZON REFUND,True


Simple extraction marker coverage: 100.0%


## 4. Download the Qwen GGUF model

This downloads a Q4_K_M quantized model from the official Qwen GGUF repository. It may take a few minutes and needs internet access once.

If you have already downloaded a compatible Qwen3-1.7B Q4_K_M GGUF, set `MODEL_PATH` manually and skip the download.

**Do not use a cloud inference API.** The download is only for obtaining the model weights; the following inference cell runs locally.


In [7]:
from pathlib import Path
from huggingface_hub import hf_hub_download

# Download Qwen3 1.7B, 4-bit quantized model
repo_id = "Antigma/Qwen3-1.7B-GGUF"
filename = "qwen3-1.7b-q4_k_m.gguf"

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading Qwen3-1.7B Q4_K_M...")

downloaded_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    local_dir=MODEL_DIR
)

MODEL_PATH = Path(downloaded_path)

print("\nDownload complete!")
print("Model path:", MODEL_PATH.resolve())
print(f"Model size: {MODEL_PATH.stat().st_size / (1024**3):.2f} GiB")

assert MODEL_PATH.exists(), "Model file was not downloaded!"


Download complete!
Model path: D:\FinGPT\models\qwen3-1.7b-q4_k_m.gguf
Model size: 1.19 GiB


## 5. Load Qwen on CPU

We explicitly set `n_gpu_layers=0`. The GGUF file contains quantized weights; Q4_K_M reduces RAM use compared with full precision.

We also limit context and output tokens for this small document task. If your machine becomes unresponsive, lower `n_threads` or close other applications.


In [8]:
from llama_cpp import Llama

cpu_threads = max(2, min((os.cpu_count() or 4) - 1, 8))

llm = Llama(
    model_path=str(MODEL_PATH),
    n_ctx=4096,
    n_threads=cpu_threads,
    n_batch=256,
    n_gpu_layers=0,  # CPU ONLY
    verbose=False
)

print("Qwen loaded.")
print("CPU threads:", cpu_threads)
print("GPU layers:", 0)


Qwen loaded.
CPU threads: 7
GPU layers: 0


## 6. Ask one question before building the evaluator

We provide the complete extracted statement as context. It is short enough that retrieval is unnecessary for this first baseline.

The model is instructed to answer only from the document and to avoid guessing. `/no_think` asks Qwen3 to use its non-thinking response mode, which is useful for a compact extraction-style task.


In [9]:
SYSTEM_PROMPT = (
    "You are a careful personal-finance assistant. "
    "Use only the supplied bank statement. "
    "Do not invent transactions or balances. "
    "For questions asking for an amount, calculate from the listed transactions. "
    "If the statement does not contain enough information, say so. "
    "Return a concise answer and show the relevant calculation. /no_think"
)

def ask_qwen(question: str, context: str = document_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            "BANK STATEMENT:\\n"
            + context
            + "\\n\\nQUESTION: "
            + question
        )}
    ]

    started = time.perf_counter()
    response = llm.create_chat_completion(
        messages=messages,
        temperature=0.0,
        top_p=1.0,
        max_tokens=180
    )
    elapsed = time.perf_counter() - started

    answer = response["choices"][0]["message"]["content"].strip()
    usage = response.get("usage", {})
    return {
        "answer": answer,
        "latency_seconds": elapsed,
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
    }

test_question = "How much did I spend on Amazon in October? Include debit transactions only."
first_result = ask_qwen(test_question)
print(first_result["answer"])
print(f"Latency: {first_result['latency_seconds']:.2f} seconds")


<think>

</think>

To determine how much was spent on Amazon in October, we need to identify the debit transactions under the "Amazon" description.

From the bank statement:
- **2026-10-01**: AMAZON PAY INDIA | 1,299.00 | DEBIT
- **2026-10-22**: AMAZON REFUND | 499.00 | CREDIT

Only the debit transaction under "AMAZON PAY INDIA" is considered as a spending transaction. The refund is a credit transaction and not a spending transaction.

**Total amount spent on Amazon in October (debit transactions only):** 1,299.00 INR
Latency: 31.20 seconds


## 7. Create an evaluation set with known answers

The answer key is computed from the synthetic CSV, not typed by the model. This avoids accidentally scoring the model against its own output.

For the first baseline, evaluate numeric questions:
- Amazon debit total,
- SIP/investment debit total,
- total debit amount,
- total credit amount,
- number of debit transactions,
- largest debit transaction.

This is a small smoke-test set, not a statistically robust benchmark. Tomorrow we can expand it with paraphrases, date filters, ambiguous merchants, and questions the document cannot answer.


In [10]:
def money(value):
    return round(float(value), 2)

debits = gt[gt["direction"] == "DEBIT"]
credits = gt[gt["direction"] == "CREDIT"]

gold = [
    {
        "id": "amazon_debits",
        "question": "How much did I spend on Amazon in October? Include debit transactions only.",
        "expected": money(gt[
            gt["description"].str.contains("AMAZON", case=False)
            & (gt["direction"] == "DEBIT")
        ]["amount"].sum()),
        "unit": "INR"
    },
    {
        "id": "sip_total",
        "question": "What was my total SIP and mutual fund investment debit in this statement?",
        "expected": money(gt[
            gt["description"].str.contains("SIP|MUTUAL FUND", case=False, regex=True)
            & (gt["direction"] == "DEBIT")
        ]["amount"].sum()),
        "unit": "INR"
    },
    {
        "id": "all_debits",
        "question": "What is the total amount debited from my account in this statement?",
        "expected": money(debits["amount"].sum()),
        "unit": "INR"
    },
    {
        "id": "all_credits",
        "question": "What is the total amount credited to my account in this statement?",
        "expected": money(credits["amount"].sum()),
        "unit": "INR"
    },
    {
        "id": "debit_count",
        "question": "How many debit transactions are listed?",
        "expected": int((gt["direction"] == "DEBIT").sum()),
        "unit": "COUNT"
    },
    {
        "id": "largest_debit",
        "question": "What is the largest debit transaction amount?",
        "expected": money(debits["amount"].max()),
        "unit": "INR"
    },
]

gold_df = pd.DataFrame(gold)
display(gold_df)


,id,question,expected,unit
0,amazon_debits,How much did I spend on Amazon in October? Inc...,2098.0,INR
1,sip_total,What was my total SIP and mutual fund investme...,8000.0,INR
2,all_debits,What is the total amount debited from my accou...,28518.0,INR
3,all_credits,What is the total amount credited to my accoun...,91699.0,INR
4,debit_count,How many debit transactions are listed?,9.0,COUNT
5,largest_debit,What is the largest debit transaction amount?,15000.0,INR


## 8. Extract the model's numeric answer

For this first benchmark, we score whether the answer contains the expected number.

This is intentionally a narrow metric:
- It can detect a correct amount even if the model adds explanatory text.
- It does **not** prove the explanation is correct.
- It does **not** prove the model used the right transactions.
- A model can mention the correct number by chance.

We'll add semantic and evidence-based checks in a later evaluation session.


In [11]:

import re

def parse_numbers(text: str):
    """
    Extract numbers from text.
    Examples:
    ₹2,098.00 -> 2098.00
    90,000.00 -> 90000.00
    90000 -> 90000.0
    """
    cleaned = text.replace(",", "")

    matches = re.findall(
        r"(?<![A-Za-z])\d+(?:\.\d{1,2})?",
        cleaned
    )

    values = []

    for item in matches:
        try:
            values.append(float(item))
        except ValueError:
            pass

    return values


def numeric_match(answer: str, expected, unit: str, tolerance: float = 0.01):
    values = parse_numbers(answer)

    if unit == "COUNT":
        return any(
            int(round(v)) == int(expected) and abs(v - expected) < 0.01
            for v in values
        )

    return any(
        abs(v - float(expected)) <= tolerance
        for v in values
    )


# Quick parser sanity checks
assert numeric_match("I spent ₹2,098.00 in total.", 2098.00, "INR")
assert numeric_match("There were 9 debit transactions.", 9, "COUNT")
assert numeric_match("My salary was ₹90,000.00.", 90000.00, "INR")

print("Numeric answer parser checks passed!")


Numeric answer parser checks passed!


## 9. Run the baseline benchmark

This sends each test question to the same un-fine-tuned local model. The model is not updated between questions.

We record:
- expected answer,
- model response,
- numeric match,
- latency,
- token counts when provided.


In [12]:
results = []

for item in gold:
    output = ask_qwen(item["question"])
    passed = numeric_match(
        output["answer"],
        item["expected"],
        item["unit"]
    )
    results.append({
        "id": item["id"],
        "question": item["question"],
        "expected": item["expected"],
        "answer": output["answer"],
        "numeric_match": passed,
        "latency_seconds": output["latency_seconds"],
        "prompt_tokens": output["prompt_tokens"],
        "completion_tokens": output["completion_tokens"],
    })

results_df = pd.DataFrame(results)
display(results_df[[
    "id", "expected", "answer", "numeric_match", "latency_seconds"
]])


,id,expected,answer,numeric_match,latency_seconds
0,amazon_debits,2098.0,<think>\n\n</think>\n\nTo determine how much w...,False,14.990089
1,sip_total,8000.0,<think>\n\n</think>\n\nTo find the total SIP a...,True,14.098131
2,all_debits,28518.0,<think>\n\n</think>\n\nTo find the total amoun...,False,18.792102
3,all_credits,91699.0,<think>\n\n</think>\n\nTo find the total amoun...,False,14.534617
4,debit_count,9.0,<think>\n\n</think>\n\nThe debit transactions ...,False,7.866588
5,largest_debit,15000.0,<think>\n\n</think>\n\nThe largest debit trans...,True,2.801600


## 10. Calculate baseline metrics

We report two primary metrics:

1. **Numeric exact-match accuracy:** fraction of questions where the expected number appears in the answer within tolerance.
2. **Average latency:** average wall-clock time per question on this machine.

Also report median and slowest latency. A small test set makes these numbers noisy, so treat them as a baseline, not a universal performance claim.


In [13]:
accuracy = results_df["numeric_match"].mean()
avg_latency = results_df["latency_seconds"].mean()
median_latency = results_df["latency_seconds"].median()
max_latency = results_df["latency_seconds"].max()

metrics = pd.DataFrame([{
    "model": "Qwen3-1.7B Q4_K_M (CPU)",
    "questions": len(results_df),
    "numeric_exact_match_accuracy": accuracy,
    "avg_latency_seconds": avg_latency,
    "median_latency_seconds": median_latency,
    "max_latency_seconds": max_latency,
}])

display(metrics)
print(f"Numeric exact-match accuracy: {accuracy:.1%}")
print(f"Average latency: {avg_latency:.2f}s")


,model,questions,numeric_exact_match_accuracy,avg_latency_seconds,median_latency_seconds,max_latency_seconds
0,Qwen3-1.7B Q4_K_M (CPU),6,0.333333,12.180521,14.316374,18.792102


Numeric exact-match accuracy: 33.3%
Average latency: 12.18s


## 11. Save experiment results

Save the baseline so we can compare it with tomorrow's fine-tuned model. The CSV contains only synthetic test questions and answers.


In [14]:
results_path = ROOT / "day1_baseline_results.csv"
metrics_path = ROOT / "day1_baseline_metrics.csv"

results_df.to_csv(results_path, index=False)
metrics.to_csv(metrics_path, index=False)

print("Saved:", results_path)
print("Saved:", metrics_path)


Saved: d:\FinGPT\day1_baseline_results.csv
Saved: d:\FinGPT\day1_baseline_metrics.csv


## 12. Manual review: don't trust the score blindly

Inspect every wrong answer. Classify the failure:

- **Extraction failure:** the PDF text missed or corrupted a transaction.
- **Comprehension failure:** the model misunderstood Amazon, SIP, debit/credit, etc.
- **Arithmetic failure:** it selected the right rows but summed incorrectly.
- **Instruction failure:** it ignored the requested scope or output.
- **Evaluation failure:** the numeric checker missed a correct answer or accepted a misleading one.

Tomorrow's dataset should target the actual failure modes, not random question-answer pairs.

### Day 1 exit checklist
- [ ] PDF created in `sample/` and text extracted.
- [ ] Qwen loaded with `n_gpu_layers=0`.
- [ ] At least one question answered locally.
- [ ] All six benchmark questions run.
- [ ] Accuracy and latency saved to CSV.
- [ ] Wrong answers manually classified.

### What we do on Day 2
1. Expand the synthetic statement and create 100–300 varied question/structured-intent examples.
2. Define a strict JSON schema for intents (merchant, category, direction, date range, aggregation).
3. Split examples into train/validation/test **before** fine-tuning.
4. Fine-tune a small instruction model using LoRA/QLoRA on a suitable training machine.
5. Re-run the exact same held-out evaluation questions and compare with today's baseline.

**Important:** We should fine-tune the model to interpret questions into structured filters, not to memorize bank transactions or do arithmetic. SQL/Python remains responsible for exact totals.
